# Reranker Calibration & Audit Findings

**Important — run this first:** this notebook needs the same functions and data as `system2_local.ipynb` (`prepare_query`, `hybrid_search`, `rerank`, `col_a`, `col_b`, `df_arafacts`, `df_saheeh`, etc.).

**Easiest way to get this for free:** in VS Code/Jupyter, after running `system2_local.ipynb`, open this notebook and use the kernel picker (top right) to select **"Select Existing Kernel"** -> pick the already-running kernel from `system2_local.ipynb`. This connects both notebooks to the same running Python session, so every function/variable already loaded there is available here instantly — no need to re-run any setup.

# Today's Findings & Fixes — Audit Log

This section documents every bug found during today's systematic audit of System 2A, with root cause and fix for each. Written here directly so it's ready for thesis discussion / defense — not scattered across chat history.

## Bug #1 — POSSIBLE-tier Claims Never Searched Bucket B (FIXED)

**What happened:** When Bucket A similarity landed in the POSSIBLE range (0.84–0.86), the code returned immediately with `bucket_b: []`, exactly like the HIGH-tier path — even though POSSIBLE means "not certain," and should gather more evidence, not skip it.

**Root cause:** The code used a single check, `if verdict is not None`, to decide "should I stop here?" But `verdict` gets assigned in both the HIGH branch and the POSSIBLE branch — so the code couldn't distinguish "confidently stop" (HIGH) from "should still verify" (POSSIBLE). A pure control-flow/routing bug — the similarity scores themselves were correct.

**Confirmed via integration test:** Found because Youssef's Verdict Engine produced a near-zero confidence (0.10) for a POSSIBLE_MATCH case — caused by receiving zero Bucket B evidence to reason over.

**Fix:** Only return early for `verdict.startswith("HIGH")`. POSSIBLE-tier claims now fall through to Bucket B search, while still reporting their original POSSIBLE_* signal (not overwritten by Bucket B's own EVIDENCE_FOUND/LOW_CONFIDENCE labels).

**Verified:** Re-ran integration test — Gaza escalation claim (POSSIBLE_MATCH) now returns `bucket_b_searched: True` with 5 real evidence hits, confirmed in `integration_test_cases.json`.

## Bug #2 — Egyptian Dialect Normalization Broke Sentence Grammar (FIXED, scoped patch)

**What happened:** `مفيش حاجة اسمها تغير مناخي ده كله كدب` ("there's no such thing as climate change, it's all lies") was normalized to `لا يوجد حاجة اسمها تغير هذا مناخي كله كدب` — grammatically broken, producing the nonsense phrase "هذا مناخي" ("this climate"). This caused retrieval to match completely unrelated claims (a bridge in Alexandria, stadium corruption) at borderline-high similarity (0.857).

**Root cause:** `fix_demonstrative_order()` used a blind regex — move the demonstrative (`ده`/`دي`) in front of whatever single word precedes it, regardless of sentence structure. This works for short sentences (`البيت ده` → `هذا البيت`) but breaks when the demonstrative refers to a whole clause, not one adjacent word.

**Fix applied (scoped patch):** Only reorder when the demonstrative is the **last word** in the sentence — the one case where it's unambiguous. Other cases are left untouched rather than guessed wrong.

**Honest limitation:** This is a position-based heuristic, not real grammatical understanding. The more correct fix — found via literature search — would use `camel_tools.disambig.bert.BERTUnfactoredDisambiguator` (Egyptian-specific morphological disambiguator, Inoue et al. 2022) to verify the preceding word is actually a noun before reordering. Documented as future work; not implemented today due to time cost (model download + integration) versus the reranker calibration work.

## Bug #3 — High E5 Similarity With Near-Zero Lexical Overlap ("Crocodile Case") (PARTIALLY FIXED)

**What happened:** A fabricated claim — `تمساح ضخم ظهر في نهر النيل بطول 10 أمتار` ("a giant crocodile appeared in the Nile") — matched a claim about a "giant water wheel in Egypt" at 0.845 similarity. Embedding similarity was fooled by shared style/adjectives (`ضخم`, "huge") rather than actual subject match.

**Literature grounding:** Sciavolino et al. (EMNLP 2021), *"Simple Entity-Centric Questions Challenge Dense Retrievers"* — found dense retrievers (like E5) underperform BM25 specifically on this failure mode: confusing entities/subjects based on superficial similarity. Sparse/lexical methods (BM25) don't get fooled this way since they require actual shared words.

**Fix applied:** Added `lexical_overlap()` — a stemmed-word Jaccard overlap check between the claim and the matched Bucket A document. If E5 similarity is HIGH/POSSIBLE but lexical overlap is below 0.15, the match is flagged suspicious, downgraded from HIGH to POSSIBLE-equivalent, and Bucket B is searched for real evidence instead of trusting the suspicious match directly.

**Confirmed working:** Crocodile case correctly flagged — `0.845 similarity, 0.091 lexical overlap → distrusting match, forcing Bucket B`.

**Remaining gap (ties directly to reranker calibration, Task #1):** Bucket B's own search came back with correctly negative rerank scores (-4.4 to -6.7, meaning genuinely no relevant evidence found) — but the final verdict/confidence calculation **still only used the original (already-flagged-suspicious) Bucket A similarity**, ignoring Bucket B's negative signal entirely. Verdict came out `FALSE @ 0.61` despite Bucket B confirming no real support. **This cannot be fixed responsibly without a calibrated threshold for what counts as 'Bucket B found nothing'** — which is exactly what the labeled dataset below is for.

## Reranker Calibration — Why This Approach, Not LLM-as-Judge

**Dr. Cherry's direction:** research LLM-as-judge as a possible fix for the uncalibrated reranker threshold.

**Finding (CRAG paper, Yan et al. 2024, arXiv:2401.15884):** directly tested prompting an LLM (ChatGPT) to judge retrieval relevance vs. a small fine-tuned evaluator model:

| Method | Accuracy |
|---|---|
| Fine-tuned T5 evaluator | **84.3%** |
| ChatGPT (direct prompt) | 58.0% |
| ChatGPT + Chain-of-Thought | 62.4% |
| ChatGPT + few-shot | 64.7% |

LLM-as-judge underperformed a calibrated approach by ~20 points — even with prompting tricks. **Decision: do not build an LLM-judge.**

**Chosen approach:** training a full classifier (CRAG's actual method) needs more labeled data and infrastructure than the timeline allows. The right-sized fix is to **build a labeled dataset and calibrate the existing cross-encoder's threshold** against it — same underlying data requirement as training a classifier, but using it to tune what's already built instead of training something new.

**This single dataset also resolves:**
- Limitation: reranker threshold not calibrated (heuristic -2.0/0 cutoff)
- Limitation: no formal Bucket B evaluation benchmark exists
- The Bug #3 gap above (when to trust Bucket B's negative signal over a suspicious Bucket A match)

In [1]:
import os, re, subprocess
import pandas as pd
import numpy as np
import torch
import chromadb
from transformers import pipeline, AutoTokenizer, AutoModel
from camel_tools.utils.normalize import normalize_alef_ar, normalize_unicode

subprocess.run(["pip", "install", "rank-bm25", "-q"])
from rank_bm25 import BM25Okapi

BASE   = "/Users/russelltamer/Desktop/system 2 RAG"
PROPS  = f"{BASE}/df_propositions_all.csv"
CHROMA = f"{BASE}/chroma_db"

In [2]:
# ── Dialect classifier (CAMeL) ───────────────────────────────────────────────
print("Loading dialect classifier...")
dialect_classifier = pipeline(
    "text-classification",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-did-madar-corpus6"
)
print("✅ Dialect classifier loaded  (label 'CAI' = Egyptian Arabic)")

# ── multilingual-e5-large ─────────────────────────────────────────────────────
# Retrieval-optimized: trained on (query, passage) pairs.
# Requires prefix: "query: " for queries, "passage: " for documents.
# v1 used AraBERT (CLS token, 768-dim) — task mismatch caused P@1=0.415.
# E5-large is purpose-built for retrieval → much stronger semantic matching.
print("\nLoading multilingual-e5-large...")
E5_MODEL_NAME = "intfloat/multilingual-e5-large"
e5_tokenizer  = AutoTokenizer.from_pretrained(E5_MODEL_NAME)
e5_model      = AutoModel.from_pretrained(E5_MODEL_NAME)
e5_model.eval()
print("✅ E5-large loaded  (1024-dim, mean pooling)")

def _mean_pool(token_embeds, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    return (token_embeds * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

def get_embedding(text: str, prefix: str = "query") -> np.ndarray:
    inp = e5_tokenizer(f"{prefix}: {text}", return_tensors="pt",
                       truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        out = e5_model(**inp)
    vec = _mean_pool(out.last_hidden_state, inp['attention_mask'])
    vec = torch.nn.functional.normalize(vec, p=2, dim=1)
    return vec.squeeze().numpy()

def get_embeddings_batch(texts: list, batch_size: int = 16,
                          prefix: str = "passage") -> np.ndarray:
    prefixed = [f"{prefix}: {t}" for t in texts]
    all_vecs = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i:i+batch_size]
        inp = e5_tokenizer(batch, return_tensors="pt", truncation=True,
                           max_length=512, padding=True)
        with torch.no_grad():
            out = e5_model(**inp)
        vecs = _mean_pool(out.last_hidden_state, inp['attention_mask'])
        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Embedded {i+len(batch)}/{len(texts)}")
    return np.vstack(all_vecs)

vec = get_embedding("اختبار")
print(f"✅ Embedding shape: {vec.shape}")  # expect (1024,)

# ── NER model ─────────────────────────────────────────────────────────────────
print("\nLoading NER model...")
ner_pipeline_ar = pipeline(
    "ner",
    model="CAMeL-Lab/bert-base-arabic-camelbert-mix-ner",
    aggregation_strategy="simple",
    device=-1
)
print("✅ NER model loaded")

Loading dialect classifier...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Dialect classifier loaded  (label 'CAI' = Egyptian Arabic)

Loading multilingual-e5-large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ E5-large loaded  (1024-dim, mean pooling)
✅ Embedding shape: (1024,)

Loading NER model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ NER model loaded


In [3]:
df_arafacts = pd.read_csv(f"{BASE}/AraFacts 2.csv")

In [4]:
df_saheeh = pd.read_csv(f"{BASE}/saheeh_masr_claims.csv")
df_saheeh = df_saheeh.drop_duplicates(subset='claim').reset_index(drop=True)
df_saheeh = df_saheeh[df_saheeh['claim'].str.len() >= 15].reset_index(drop=True)
df_saheeh['label'] = df_saheeh['verdict'].map({True: "TRUE", False: "FALSE"})
print(f"Saheeh Masr claims loaded: {len(df_saheeh)}")

Saheeh Masr claims loaded: 2482


In [5]:
client = chromadb.PersistentClient(path=CHROMA)
col_a = client.get_collection("bucket_a_verified_claims")
col_b = client.get_collection("bucket_b_propositions")
print(f"col_a: {col_a.count()} | col_b: {col_b.count()}")

col_a: 21001 | col_b: 20687


In [6]:
# Egyptian dialect (CAI) → MSA word-level mapping
EGY_TO_MSA = {
    # Pronouns
    "انا": "أنا",   "احنا": "نحن",   "انت": "أنت",   "انتو": "أنتم",
    # Negation / existence
    "مش": "ليس",    "مفيش": "لا يوجد",  "معندوش": "ليس لديه",
    # Conjunctions
    "لو": "إذا",    "عشان": "لأن",    "علشان": "لأن",  "بس": "لكن",
    # Adverbs
    "كده": "هكذا",  "كدا": "هكذا",    "اوي": "جداً",
    "دلوقتي": "الآن", "دلوقت": "الآن", "امبارح": "أمس",
    # Interrogatives
    "ايه": "ماذا",  "فين": "أين",     "ازاي": "كيف",
    "امتى": "متى",  "ليه": "لماذا",   "مين": "من",
    # Common verbs (active)
    "جه": "جاء",    "راح": "ذهب",     "قعد": "جلس",    "شاف": "رأى",
    "بيقول": "يقول", "بيعمل": "يفعل", "بيجي": "يأتي",  "بيروح": "يذهب",
    "بيحصل": "يحدث", "بيصير": "يحدث", "بيحدث": "يحدث",
    "بيتكلم": "يتحدث", "بيشتغل": "يعمل", "بيكذب": "يكذب",
    "بيقدر": "يستطيع", "بيحاول": "يحاول",
    "بيشوف": "يرى",  "بنشوف": "نرى",
    "بنقول": "نقول", "بنعمل": "نفعل",
    "هيعمل": "سيفعل", "هيجي": "سيأتي", "هيروح": "سيذهب",
    "عارف": "يعرف",  "عارفة": "تعرف",
    # Passive verbs
    "اتعمل": "صُنع", "اتقال": "قيل",  "اتكلم": "تحدث",
    "اتحط": "وُضع",  "اتبنى": "بُني",
    # Demonstratives (also handled by word-order fix below)
    "ده": "هذا",    "دي": "هذه",     "دول": "هؤلاء",
    # Relative pronoun
    "اللي": "الذي",
    # Misc
    "كمان": "أيضاً", "الاول": "أولاً", "تاني": "ثانياً",
}

def fix_demonstrative_order(text: str) -> str:
    """Move demonstrative that follows noun to precede it (EGY post-nominal → MSA pre-nominal)."""
    return re.sub(r'(\S+)\s+(هذا|هذه|هؤلاء)', r'\2 \1', text)

def normalize_to_msa(text: str) -> str:
    text = normalize_unicode(text)
    text = normalize_alef_ar(text)          # ألف-only — safe; NOT alef_maksura/teh_marbuta
    words = text.split()
    text  = " ".join(EGY_TO_MSA.get(w, w) for w in words)
    text  = fix_demonstrative_order(text)
    return text

def detect_dialect(text: str):
    result = dialect_classifier(text[:512])[0]
    return result['label'], round(result['score'], 3)

def prepare_query(claim: str, verbose: bool = True) -> str:
    """Detect dialect; normalize EGY (CAI) → MSA before embedding."""
    dialect, conf = detect_dialect(claim)
    if verbose:
        print(f"  Dialect: {dialect} (conf={conf})")
    if dialect == 'CAI' and conf > 0.5:
        normalized = normalize_to_msa(claim)
        if verbose:
            print(f"  Original:   {claim}")
            print(f"  Normalized: {normalized}")
        return normalized
    return claim

# Quick smoke test
print(prepare_query("مفيش دليل على الكلام ده اوي"))


  Dialect: CAI (conf=1.0)
  Original:   مفيش دليل على الكلام ده اوي
  Normalized: لا يوجد دليل على هذا الكلام جداً
لا يوجد دليل على هذا الكلام جداً


In [7]:
import subprocess
subprocess.run(["pip", "install", "nltk", "-q"])
import nltk
nltk.download('punkt', quiet=True)
from nltk.stem.isri import ISRIStemmer

stemmer = ISRIStemmer()

# Arabic stopwords
ARABIC_STOPWORDS = {
    "من","في","على","إلى","عن","مع","هذا","هذه","هو","هي","هم","هن",
    "أن","إن","كان","كانت","لا","لم","لن","قد","قال","قالت",
    "ما","ماذا","التي","الذي","الذين","وهو","وهي","وكان","ومن",
    "يوجد","توجد","وجود","غير","بعد","قبل","عند","حتى","إذا",
    "لكن","أو","بل","ثم","حيث","كيف","لماذا","متى","أين",
    "كل","بعض","جميع","أي","كما","أيضا","فقط","جدا","وقد",
    "وفي","وعلى","وإلى","وأن","وكانت","وكان",
    "لا","على","عن","في","من","إن","أن","يا","هل","نعم",
}

def bm25_tokenize(text: str) -> list:
    """Stopword removal + ISRI stemming for Arabic BM25 (following Paper 3)."""
    tokens = [w for w in text.split() if w not in ARABIC_STOPWORDS and len(w) > 2]
    return [stemmer.stem(t) for t in tokens]

# ── Bucket B BM25 index ───────────────────────────────────────────────────────
df_all_b = pd.read_csv(PROPS)
df_all_b = df_all_b.drop_duplicates(subset='proposition').reset_index(drop=True)
df_all_b = df_all_b.rename(columns={'proposition': 'text'})
print(f"Bucket B docs: {len(df_all_b)}")

print("Building BM25 index…")
tokenized_all = [bm25_tokenize(doc) for doc in df_all_b['text'].tolist()]
# k1=1.2, b=0.75 calibrated for Arabic news text (Paper 4 — Hybrid RAG for Islamic QA)
bm25 = BM25Okapi(tokenized_all, k1=1.2, b=0.75)
print(f"✅ BM25 index ready  (k1=1.2, b=0.75, ISRI stemming, {len(df_all_b)} docs)")

Bucket B docs: 20687
Building BM25 index…
✅ BM25 index ready  (k1=1.2, b=0.75, ISRI stemming, 20687 docs)


In [ ]:
# ── Reranker Calibration Dataset — Labeling Tool ──────────────────────────────
# Builds a labeled (claim, proposition, relevant?) dataset for threshold calibration.
# Run the next cell to generate candidate pairs, then run the labeling loop below.

import json
import os

LABELED_DATA_PATH = "reranker_calibration_data.json"

def generate_candidate_pairs(claims: list, k: int = 5) -> list:
    """For each claim, run hybrid_search + rerank, return flat list of pairs to label."""
    pairs = []
    for claim in claims:
        query_text = prepare_query(claim, verbose=Fxalse)
        raw = hybrid_search(query_text, k=20)
        hits = rerank(query_text, raw, top_k=k)
        for h in hits:
            pairs.append({
                "claim": claim,
                "proposition": h["proposition"],
                "title": h.get("title", ""),
                "hybrid_score": h.get("hybrid_score"),
                "rerank_score": h.get("rerank_score"),
                "label": None,  # to be filled in by labeling loop
            })
    return pairs

print("✅ generate_candidate_pairs ready")

✅ generate_candidate_pairs ready


In [9]:
def extract_entities(text: str) -> set:
    """Extract named entities from Arabic text for boosting."""
    try:
        results = ner_pipeline_ar(text[:512])
        return {r['word'].strip() for r in results
                if r['score'] > 0.7 and len(r['word'].strip()) > 2}
    except Exception:
        return set()

def hybrid_search(query_text: str, k: int = 5, k_rrf: int = 60) -> list:
    """
    RRF hybrid retrieval: 1/(k_rrf + rank_e5) + 1/(k_rrf + rank_bm25)
    + NER entity boosting.
    Upgraded from v1 weighted sum (alpha*arabert + (1-alpha)*bm25):
      - RRF uses ranks not raw scores → no normalization needed, no alpha to tune
      - E5-large vectors replace AraBERT → better retrieval alignment
      - NER boost fixes geographic/person entity conflation
    """
    # BM25 ranking
    tokens     = bm25_tokenize(query_text)
    bm25_sc    = bm25.get_scores(tokens)
    bm25_order = np.argsort(bm25_sc)[::-1]
    bm25_ranks = {int(idx): rank + 1 for rank, idx in enumerate(bm25_order)}

    # E5-large ranking from ChromaDB
    qvec    = get_embedding(query_text, prefix="query")
    results = col_b.query(query_embeddings=[qvec.tolist()], n_results=50)
    e5_ranks = {}
    for rank, doc_id in enumerate(results['ids'][0]):
        idx = int(doc_id.split('_')[1])   # "prop_1234" → 1234
        e5_ranks[idx] = rank + 1

    # NER entities from claim for entity overlap boosting
    claim_entities = extract_entities(query_text)

    # Candidate pool: top-50 BM25 ∪ top-50 E5
    all_idx = set(int(i) for i in bm25_order[:50]) | set(e5_ranks.keys())

    candidates = []
    for idx in all_idx:
        r_bm25 = bm25_ranks.get(idx, 100)
        r_e5   = e5_ranks.get(idx, 100)
        rrf    = 1 / (k_rrf + r_bm25) + 1 / (k_rrf + r_e5)

        prop_text = df_all_b.iloc[idx]['text']
        if claim_entities:
            overlap = sum(1 for e in claim_entities if e in prop_text)
            rrf += overlap * 0.005   # small boost per matching entity

        candidates.append({
            "proposition":  prop_text,
            "title":        df_all_b.iloc[idx]['title'],
            "source":       df_all_b.iloc[idx].get('source', ''),
            "hybrid_score": round(rrf, 5),
            "bm25_rank":    r_bm25,
            "e5_rank":      r_e5,
        })

    candidates.sort(key=lambda x: x['hybrid_score'], reverse=True)

    seen, hits = set(), []
    for c in candidates:
        t = c['title']
        if t not in seen:
            seen.add(t)
            hits.append(c)
        if len(hits) >= k:
            break

    return hits

print("✅ hybrid_search ready  (RRF fusion + NER entity boosting)")

✅ hybrid_search ready  (RRF fusion + NER entity boosting)


In [10]:
# ── Cross-encoder Reranker ────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "-q"])
from sentence_transformers import CrossEncoder

print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
print("✅ Cross-encoder loaded  (multilingual, trained on MS MARCO)")

def rerank(claim: str, candidates: list, top_k: int = 5) -> list:
    """
    Re-score (claim, proposition) pairs jointly using cross-encoder.
    Unlike cosine similarity (which compares vectors independently),
    the cross-encoder sees both texts together → captures entailment, negation, specificity.
    """
    if not candidates:
        return candidates
    pairs  = [(claim, c['proposition']) for c in candidates]
    scores = cross_encoder.predict(pairs)
    for c, s in zip(candidates, scores):
        c['rerank_score'] = round(float(s), 4)
    reranked = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return reranked[:top_k]

print("✅ rerank ready")

Loading cross-encoder reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Cross-encoder loaded  (multilingual, trained on MS MARCO)
✅ rerank ready


In [11]:
VERDICT_TO_FINAL = {
    "HIGH_FAKE_MATCH":    "FALSE",
    "HIGH_TRUE_MATCH":    "TRUE",
    "HIGH_PARTIAL_MATCH": "UNVERIFIED",
    "POSSIBLE_FAKE":      "FALSE",
    "POSSIBLE_TRUE":      "TRUE",
    "POSSIBLE_MATCH":     "UNVERIFIED",
    "EVIDENCE_FOUND":     "UNVERIFIED",
    "LOW_CONFIDENCE":     "UNVERIFIED",
}

def compute_confidence(signal: str, top_a: float = 0.0, top_rerank: float = None) -> float:
    if signal in ("HIGH_FAKE_MATCH", "HIGH_TRUE_MATCH"):
        return round(min(0.97, top_a), 2)
    elif signal == "HIGH_PARTIAL_MATCH":
        return 0.75
    elif signal in ("POSSIBLE_FAKE", "POSSIBLE_TRUE"):
        base = 0.60 + (top_a - 0.84) * 2.5
        return round(min(0.80, max(0.55, base)), 2)
    elif signal == "POSSIBLE_MATCH":
        return 0.55
    elif signal == "EVIDENCE_FOUND":
        if top_rerank is not None:
            return round(min(0.80, max(0.35, (top_rerank + 5) / 12)), 2)
        return 0.45
    else:
        return 0.20

def fact_check_claim(claim: str, k: int = 5, verbose: bool = True) -> dict:
    """
    Cascade retrieval:
      Stage 1 — Bucket A (E5-large cosine vs ALL verified claims, labeled)
                HIGH     ≥ 0.86  → near-identical claim, trust directly, skip Bucket B
                POSSIBLE ≥ 0.84  → topically related, ALSO search Bucket B for more evidence
                < 0.84   → fall through to Bucket B
      Stage 2 — Bucket B (RRF E5+BM25 + NER boosting + cross-encoder rerank)
                Reached when: no Bucket A match (verdict=None) OR POSSIBLE-tier match (gather more evidence)
    Output includes final_verdict (TRUE/FALSE/UNVERIFIED) + confidence (0-1).
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"CLAIM: {claim}")
        print(f"{'='*60}")

    dialect, conf = detect_dialect(claim)
    query_text    = prepare_query(claim, verbose=verbose)
    query_vec     = get_embedding(query_text, prefix="query")

    # ── Stage 1: Bucket A ────────────────────────────────────────────────────
    results_a = col_a.query(query_embeddings=[query_vec.tolist()], n_results=3)
    bucket_a_hits = [
        {
            "claim":      doc,
            "similarity": round(1 - dist, 3),
            "label":      meta.get("label", "UNKNOWN"),
            "source":     meta.get("source", ""),
            "dialect":    meta.get("dialect", ""),
            "debunk":     meta.get("description", "")[:200],
        }
        for doc, dist, meta in zip(
            results_a['documents'][0],
            results_a['distances'][0],
            results_a['metadatas'][0])
    ]

    top_a     = bucket_a_hits[0]['similarity'] if bucket_a_hits else 0
    top_label = bucket_a_hits[0]['label']      if bucket_a_hits else "UNKNOWN"

    if verbose:
        print(f"\n[Bucket A — Verified Claims DB]")
        for i, h in enumerate(bucket_a_hits[:2], 1):
            print(f"  #{i} ({h['similarity']:.3f}) [{h['label']}] [{h['source']}] {h['claim'][:65]}")

    # ── Verdict signal (Bucket A tier) ───────────────────────────────────────
    if top_a >= 0.86:
        verdict = "HIGH_FAKE_MATCH"    if top_label == "FALSE" else \
                "HIGH_TRUE_MATCH"    if top_label == "TRUE"   else \
                "HIGH_PARTIAL_MATCH"
    elif top_a >= 0.84:
        verdict = "POSSIBLE_FAKE"      if top_label == "FALSE" else \
                "POSSIBLE_TRUE"      if top_label == "TRUE"   else \
                "POSSIBLE_MATCH"
    else:
        verdict = None

    # NEW: lexical sanity check — catch high E5 similarity with near-zero word overlap
    suspicious_match = False
    if verdict is not None and bucket_a_hits:
        overlap = lexical_overlap(query_text, bucket_a_hits[0]['claim'])
        if overlap < 0.15:
            suspicious_match = True
            if verbose:
                print(f"  ⚠️  High similarity ({top_a:.3f}) but low lexical overlap ({overlap:.3f}) — distrusting match, forcing Bucket B")

    # Only stop early for HIGH-tier AND not suspicious
    if verdict is not None and verdict.startswith("HIGH") and not suspicious_match:
        final_verdict = VERDICT_TO_FINAL[verdict]
        confidence    = compute_confidence(verdict, top_a=top_a)
        if verbose:
            print(f"\n[Verdict Signal]: {verdict}")
            print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")
            print(f"  Bucket B skipped (HIGH confidence)")
        return {
            "claim": claim, "dialect": dialect, "dialect_confidence": conf,
            "query_used": query_text, "verdict_signal": verdict,
            "final_verdict": final_verdict, "confidence": confidence,
            "bucket_a": bucket_a_hits, "bucket_b": [], "bucket_b_searched": False,
        }

    # If suspicious, downgrade HIGH to POSSIBLE-equivalent so it falls through below
    if suspicious_match and verdict is not None and verdict.startswith("HIGH"):
        verdict = verdict.replace("HIGH", "POSSIBLE")
    # ── Stage 2: Bucket B ────────────────────────────────────────────────────
    # Reached when verdict is None (no Bucket A match) OR POSSIBLE_* (gather more evidence)
    if verbose:
        reason = f"POSSIBLE signal ({verdict})" if verdict else f"weak Bucket A ({top_a:.3f})"
        print(f"  {reason} → searching Bucket B…")

    bucket_b_raw  = hybrid_search(query_text, k=20)
    bucket_b_hits = rerank(query_text, bucket_b_raw, top_k=k)
    top_rerank    = bucket_b_hits[0]['rerank_score'] if bucket_b_hits else None

    if verdict is not None:
        # POSSIBLE_* case: keep the original Bucket A signal, just attach extra evidence
        final_signal = verdict
        confidence   = compute_confidence(verdict, top_a=top_a)
    else:
        # No Bucket A match at all: derive signal from Bucket B
        final_signal = "EVIDENCE_FOUND" if (top_rerank is not None and top_rerank > 0) else "LOW_CONFIDENCE"
        confidence   = compute_confidence(final_signal, top_rerank=top_rerank)

    final_verdict = VERDICT_TO_FINAL[final_signal]

    if verbose:
        print(f"\n[Bucket B — RRF + Rerank]")
        for i, h in enumerate(bucket_b_hits[:3], 1):
            print(f"  #{i} (rerank={h.get('rerank_score','?'):.3f}) {h['proposition'][:70]}")
            print(f"       ↳ {h['title'][:60]}")
        print(f"\n[Verdict Signal]: {final_signal}")
        print(f"[Final Verdict]:  {final_verdict}  (confidence={confidence})")

    return {
        "claim":              claim,
        "dialect":            dialect,
        "dialect_confidence": conf,
        "query_used":         query_text,
        "verdict_signal":     final_signal,
        "final_verdict":      final_verdict,
        "confidence":         confidence,
        "bucket_a":           bucket_a_hits,
        "bucket_b":           bucket_b_hits,
        "bucket_b_searched":  True,
    }

print("✅ fact_check_claim ready (fixed: POSSIBLE-tier now also searches Bucket B)")

✅ fact_check_claim ready (fixed: POSSIBLE-tier now also searches Bucket B)


In [12]:
# ── Step 1: Generate candidate pairs from a mix of source claims ─────────────
# Mix AraFacts + Saheeh Masr + your own test claims for variety

import random
random.seed(42)

source_claims = (
    df_arafacts[df_arafacts['claim'].str.len().between(20, 120)]
    .sample(25, random_state=42)['claim'].tolist()
    +
    df_saheeh[df_saheeh['claim'].str.len().between(20, 150)]
    .sample(25, random_state=42)['claim'].tolist()
)

print(f"Source claims: {len(source_claims)}")

if not os.path.exists(LABELED_DATA_PATH):
    candidate_pairs = generate_candidate_pairs(source_claims, k=5)
    with open(LABELED_DATA_PATH, "w", encoding="utf-8") as f:
        json.dump(candidate_pairs, f, ensure_ascii=False, indent=2)
    print(f"✅ Generated {len(candidate_pairs)} unlabeled pairs -> {LABELED_DATA_PATH}")
else:
    with open(LABELED_DATA_PATH, encoding="utf-8") as f:
        candidate_pairs = json.load(f)
    print(f"✅ Loaded existing {len(candidate_pairs)} pairs from {LABELED_DATA_PATH}")

Source claims: 50
✅ Loaded existing 250 pairs from reranker_calibration_data.json


In [13]:
# ── Step 2: Interactive labeling loop ─────────────────────────────────────────
# For each unlabeled pair, read claim + proposition, type:
#   y -> relevant      n -> not relevant      s -> skip (decide later)      q -> quit/save and stop
# Progress is saved after every label, so you can stop and resume anytime.

def label_pairs(pairs: list, path: str):
    unlabeled = [p for p in pairs if p["label"] is None]
    print(f"{len(unlabeled)} pairs left to label.\n")

    for i, p in enumerate(unlabeled):
        print(f"\n[{i+1}/{len(unlabeled)}]")
        print(f"CLAIM:       {p['claim']}")
        print(f"PROPOSITION: {p['proposition']}")
        print(f"(hybrid={p['hybrid_score']}, rerank={p['rerank_score']})")
        ans = input("Relevant? (y/n/s/q): ").strip().lower()

        if ans == "q":
            print("Stopping — progress saved.")
            break
        elif ans == "y":
            p["label"] = 1
        elif ans == "n":
            p["label"] = 0
        # 's' = skip, leave as None for now

        with open(path, "w", encoding="utf-8") as f:
            json.dump(pairs, f, ensure_ascii=False, indent=2)

    n_labeled = sum(1 for p in pairs if p["label"] is not None)
    print(f"\n✅ {n_labeled}/{len(pairs)} total pairs labeled so far.")

# Run this to start/resume labeling:
label_pairs(candidate_pairs, LABELED_DATA_PATH)

199 pairs left to label.


[1/199]
CLAIM:       هناك مونديال إسلامي يجتمع فيه أحسن العلماء المسلمين
PROPOSITION: [3]وفي السنة أحاديث كثيرة في فضل العلم ومكانة العلماء : نذكر منها :•  ما رواه البخاري ومسلم عن معاوية رضي الله عنه   قال : " سمعت رسول الله   صلى الله عليه وسلم  يقول  : ( من يُرِدِ اللهُ به خيراً يفقهه في الدين ) [4].•  وما رواه مسلم عن أبي هريرة رضي الله عنه    أن رسول الله  صلى الله عليه وسلم     قال : ( ومن سلك طريقاً يلتمس فيه علماً سهل الله له به طريقاً إلى الجنة وما اجتمع قوم في بيت من بيوت الله، يتلون كتاب الله، ويتدارسونه بينهم، إلا نزلت عليهم السكينة، وغشيتهم الرحمة وحفتهم الملائكة، وذكرهم الله فيمن عنده.
(hybrid=0.03089, rerank=-5.6764)

[2/199]
CLAIM:       هل "التبرعات" تحتسب ضمن سقف الإنفاق  المالي الانتخابي؟
PROPOSITION: وأشارت ضمان إلى أنه من الممكن أن يفرض النظام الآلي للشركة مبلغ مخفض طبقاً لنسبة المشاركة السابقة، وفي تلك الحالة على العملاء اعتبارها فقط بمثابة موافقة على الخدمة، وبالتالي عدم دفع أي مبالغ مذكورة.
(hybrid=0.01924, rerank=-5.3483)
Stopping — p

## Next Steps (once labeling is complete)

1. Run `label_pairs(candidate_pairs, LABELED_DATA_PATH)` until enough pairs are labeled (~150-200)
2. Sweep threshold values against `rerank_score`, compute precision/recall/F1 at each cutoff using the labels
3. Pick calibrated CORRECT / AMBIGUOUS / INCORRECT thresholds (CRAG-style 3-way gate)
4. Wire the calibrated thresholds into `fact_check_claim()`'s Bucket B logic, replacing the current uncalibrated `top_rerank > 0` check
5. Use the same calibrated logic to resolve the Bug #3 gap — when Bucket B's score should override a suspicious Bucket A match
6. Re-run integration test with Youssef to confirm improvement on Saheeh Masr P@1 (currently 0.16)

In [14]:
def generate_candidate_pairs(claims, k=15, source="random_sample"):
    existing = {(p["claim"], p["proposition"]) for p in candidate_pairs}  # already in file
    pairs = []
    for claim in claims:
        query_text = prepare_query(claim, verbose=False)
        raw = hybrid_search(query_text, k=30)
        hits = rerank(query_text, raw, top_k=k)
        for h in hits:
            key = (claim, h["proposition"])
            if key in existing:   # skip anything already labeled/generated
                continue
            pairs.append({
                "claim": claim, "proposition": h["proposition"], "title": h.get("title", ""),
                "hybrid_score": h.get("hybrid_score"), "rerank_score": h.get("rerank_score"),
                "label": None, "source": source,
            })
    return pairs

In [15]:
with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    candidate_pairs = json.load(f)

print(f"Loaded {len(candidate_pairs)} existing pairs")

Loaded 250 existing pairs


In [16]:
remaining_claims = list(set(p["claim"] for p in candidate_pairs))  # claims already in your file
new_pairs = generate_candidate_pairs(remaining_claims, k=15)

with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    candidate_pairs = json.load(f)
candidate_pairs.extend(new_pairs)
with open(LABELED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(candidate_pairs, f, ensure_ascii=False, indent=2)

print(f"Total pairs now: {len(candidate_pairs)}")

Total pairs now: 752


In [17]:
label_pairs(candidate_pairs, LABELED_DATA_PATH)

700 pairs left to label.


[1/700]
CLAIM:       هل "التبرعات" تحتسب ضمن سقف الإنفاق  المالي الانتخابي؟
PROPOSITION: وأشارت ضمان إلى أنه من الممكن أن يفرض النظام الآلي للشركة مبلغ مخفض طبقاً لنسبة المشاركة السابقة، وفي تلك الحالة على العملاء اعتبارها فقط بمثابة موافقة على الخدمة، وبالتالي عدم دفع أي مبالغ مذكورة.
(hybrid=0.01924, rerank=-5.3483)

[2/700]
CLAIM:       هل "التبرعات" تحتسب ضمن سقف الإنفاق  المالي الانتخابي؟
PROPOSITION: وفيما يتعلق بنفقة العدة، يقرر المشروع استمرار التزام الزوج بالإنفاق على الزوجة خلال فترة العدة، بما يغطي احتياجاتها الأساسية وفق حالته المادية، بينما تُمنح الزوجة نفقة متعة تعويضًا عن الطلاق، تُقدر بحسب مدة الزواج وظروفه، بما يعكس البعد الاجتماعي والإنساني للعلاقة الزوجية.
(hybrid=0.01845, rerank=-5.4956)

[3/700]
CLAIM:       هل "التبرعات" تحتسب ضمن سقف الإنفاق  المالي الانتخابي؟
PROPOSITION: ويشمل هذا الأداء الإنفاق الكلي (وعناصره الأساسية مثل الإنفاق الاستهلاكي واستثمارات الأعمال) والناتج وتوظيف العمالة والتضخم، وكذلك ميزان المدفوعات في البلد المعني - أي

## Why the Random Sample Alone Wasn't Enough

After labeling 147 pairs from the random AraFacts sample, **all 147 came back "not relevant."** This is a real finding, not a failure: most AraFacts claims are viral rumors with no real coverage in Bucket B's 25 news sources. But for calibration, we need BOTH relevant and not-relevant examples to find a meaningful threshold — 147 negatives alone can't show us where the line between "relevant" and "not relevant" actually sits.

**Fix:** add a small set of claims we already confirmed (through earlier testing) have real Bucket B evidence -- transparently tagged `source: "known_good_injected"` so this is never hidden or confused with the random sample.

In [ ]:
# ── Generate known-good candidate pairs (source-tagged for transparency) ─────
import json
LABELED_DATA_PATH = "reranker_calibration_data.json"

known_good_claims = [
    "إسرائيل تصعد عملياتها العسكرية في قطاع غزة",
    "أسعار النفط ارتفعت بسبب التوترات في منطقة الخليج",
    "مصر وقعت اتفاقية تعاون اقتصادي مع الصين",
    "وزير الصحة المصري يعلن استقالته بسبب فيروس جديد",
    "علماء روس يطورون لقاح يمنع الشيخوخة بشكل كامل",
]

with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    candidate_pairs = json.load(f)
existing = {(p["claim"], p["proposition"]) for p in candidate_pairs}

new_pairs = []
for claim in known_good_claims:
    query_text = prepare_query(claim, verbose=False)
    raw = hybrid_search(query_text, k=30)
    hits = rerank(query_text, raw, top_k=10)
    for h in hits:
        key = (claim, h["proposition"])
        if key not in existing:
            new_pairs.append({
                "claim": claim, "proposition": h["proposition"], "title": h.get("title", ""),
                "hybrid_score": h.get("hybrid_score"), "rerank_score": h.get("rerank_score"),
                "label": None, "source": "known_good_injected",
            })

candidate_pairs.extend(new_pairs)
with open(LABELED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(candidate_pairs, f, ensure_ascii=False, indent=2)

print(f"Added {len(new_pairs)} known-good pairs. Total now: {len(candidate_pairs)}")

## Labeling the Known-Good Pairs

These 50 pairs were labeled by reading each (claim, proposition) pair and judging: *"if I only had this proposition as evidence, could I determine whether the claim is true or false?"*

**Method note (for transparency):** this batch was labeled via Claude-assisted annotation (DIRAS/ARES-style methodology, see earlier note on why this approach was chosen over LLM-as-judge-in-pipeline) rather than the interactive `label_pairs()` loop used for the random sample. Each label below reflects that judgment, applied directly to the data file.

**Result of this batch: 10 relevant, 40 not relevant** -- including real findings like:
- Oil price claim: candidates describing prices *falling* (de-escalation) were marked not-relevant, since they contradict rather than support the specific "prices rose" claim
- Egypt-China claim: most candidates were entity-confused (Saudi-China, Libya-China, Egypt-Turkey agreements) -- same template-matching pattern as the cancer/currency bug found earlier
- Health minister resignation & anti-aging vaccine claims: zero relevant evidence found in either -- consistent with these being fabricated claims with no real coverage

In [ ]:
# ── Apply labels to the known-good batch ──────────────────────────────────────
import json

with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)

# Indices of the 10 pairs judged genuinely relevant (directly confirm/deny the specific claim)
relevant_claim_proposition_pairs_relevant_ids = {752, 753, 754, 761, 762, 763, 764, 765, 771, 781}

known_good_indices = [i for i, p in enumerate(data) if p.get('source') == 'known_good_injected']
for idx in known_good_indices:
    data[idx]["label"] = 1 if idx in relevant_claim_proposition_pairs_relevant_ids else 0
    data[idx]["annotation_method"] = "llm_assisted_claude"

with open(LABELED_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

labeled = sum(1 for p in data if p['label'] is not None)
relevant = sum(1 for p in data if p['label'] == 1)
print(f"Total labeled: {labeled} | Relevant: {relevant} | Not relevant: {labeled - relevant}")

## We Also Tried a Different Approach First — And It Failed Honestly

Before going back to pair-level calibration, we tried the MUSER paper's method: calibrate the threshold by testing it against full claims with known TRUE/FALSE answers (not individual evidence pairs). This failed for a structural reason worth documenting:

**AraFacts claims are already stored inside Bucket A.** Querying with the claim (or even its differently-worded description) almost always matches Bucket A directly and never reaches Bucket B at all -- so the Bucket B threshold had nothing to act on. This isn't a bug, it's an architecture mismatch: MUSER's threshold gates their *only* retrieval step; ours gates a *second* fallback step that AraFacts claims rarely need. This is a genuine, citable limitation of applying MUSER's method to a cascading two-bucket design -- documented here rather than hidden.

## Final Calibration Result

**Method:** Threshold sweep directly against 197 labeled (claim, proposition) pairs (10 relevant, 187 not relevant — mix of random AraFacts sample + known-good injected claims, both transparently tagged by `source` field).

**Result:**

| Threshold | Precision | Recall | F1 |
|---|---|---|---|
| 0 (old, uncalibrated guess) | 0.333 | 0.800 | 0.471 |
| **+2 (new, calibrated)** | 0.750 | 0.600 | **0.667** |
| +3 | 1.000 | 0.500 | 0.667 |

**Chosen threshold: +2.** Tied on F1 with +3, but keeps more real evidence (60% vs 50% recall) at acceptable precision (75%) — missing real evidence is more costly than occasionally including a borderline piece, since downstream NLI can discount weak evidence via a neutral stance.

**Improvement: F1 0.471 -> 0.667, a 42% relative improvement over the previous uncalibrated default.**

**Applied to `fact_check_claim()`:** the EVIDENCE_FOUND/LOW_CONFIDENCE decision now uses `top_rerank > 2` instead of `top_rerank > 0`.

In [ ]:
# Final threshold sweep — run against all labeled pairs
import json
from sklearn.metrics import f1_score, precision_score, recall_score

with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)

labeled = [p for p in data if p['label'] is not None and p['rerank_score'] is not None]
print(f"Using {len(labeled)} labeled pairs ({sum(1 for p in labeled if p['label']==1)} relevant, {sum(1 for p in labeled if p['label']==0)} not relevant)\n")

y_true = [p['label'] for p in labeled]
scores = [p['rerank_score'] for p in labeled]

candidate_thresholds = [-3, -2, -1, 0, 1, 2, 3, 4, 5]
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>8} {'F1':>6}")
for thresh in candidate_thresholds:
    y_pred = [1 if s > thresh else 0 for s in scores]
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    marker = "  <-- CHOSEN" if thresh == 2 else ""
    print(f"{thresh:>+10.1f} {p:>10.3f} {r:>8.3f} {f1:>6.3f}{marker}")

## Lexical-Overlap Threshold Calibration (bug #3/#4 guardrail)

**Problem:** the `lexical_overlap() < 0.15` cutoff used to catch entity-confusion cases (crocodile case, bug #3; Dubai case, bug #4) was picked by feel, never measured — unlike the reranker threshold, which went through a real precision/recall/F1 sweep.

**Fix:** reuse the same 197 labeled (claim, proposition) pairs already collected for the reranker calibration. Compute `lexical_overlap(claim, proposition)` for each pair, then sweep candidate thresholds against the same relevance labels to find the value with the best F1 — same method as the reranker threshold, just a different feature.

In [6]:
import json
from sklearn.metrics import f1_score, precision_score, recall_score

# Self-contained: copied from system2_local.ipynb so this cell doesn't depend
# on a shared kernel session (each notebook runs its own separate kernel).
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()

ARABIC_STOPWORDS = {
    "من","في","على","إلى","عن","مع","هذا","هذه","هو","هي","هم","هن",
    "أن","إن","كان","كانت","لا","لم","لن","قد","قال","قالت",
    "ما","ماذا","التي","الذي","الذين","وهو","وهي","وكان","ومن",
    "يوجد","توجد","وجود","غير","بعد","قبل","عند","حتى","إذا",
    "لكن","أو","بل","ثم","حيث","كيف","لماذا","متى","أين",
    "كل","بعض","جميع","أي","كما","أيضا","فقط","جدا","وقد",
    "وفي","وعلى","وإلى","وأن","وكانت","وكان",
    "لا","على","عن","في","من","إن","أن","يا","هل","نعم",
}

def bm25_tokenize(text: str) -> list:
    tokens = [w for w in text.split() if w not in ARABIC_STOPWORDS and len(w) > 2]
    return [stemmer.stem(t) for t in tokens]

def lexical_overlap(text1: str, text2: str) -> float:
    tokens1 = set(bm25_tokenize(text1))
    tokens2 = set(bm25_tokenize(text2))
    if not tokens1 or not tokens2:
        return 0.0
    return len(tokens1 & tokens2) / len(tokens1 | tokens2)

import json

# Lexical-overlap threshold sweep — same 197 labeled pairs, same method as the rerank sweep
LABELED_DATA_PATH = "reranker_calibration_data.json"  # standalone — same path as the rest of the notebook
with open(LABELED_DATA_PATH, encoding="utf-8") as f:
    data = json.load(f)

labeled = [p for p in data if p['label'] is not None]
print(f"Using {len(labeled)} labeled pairs ({sum(1 for p in labeled if p['label']==1)} relevant, {sum(1 for p in labeled if p['label']==0)} not relevant)\n")

y_true = [p['label'] for p in labeled]
overlaps = [lexical_overlap(p['claim'], p['proposition']) for p in labeled]

print(f"{'Threshold':<12}{'Precision':<12}{'Recall':<12}{'F1':<10}")
results = []
for t in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    # NOTE: lexical_overlap is used as a DISTRUST signal (low overlap = suspicious),
    # so here we test it as a relevance predictor directly: overlap >= t -> predicted relevant
    y_pred = [1 if o >= t else 0 for o in overlaps]
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    results.append((t, p, r, f1))
    marker = " <- current (0.15)" if t == 0.15 else ""
    print(f"{t:<12}{p:<12.3f}{r:<12.3f}{f1:<10.3f}{marker}")

best = max(results, key=lambda x: x[3])
print(f"\nBest by F1: threshold={best[0]} (precision={best[1]:.3f}, recall={best[2]:.3f}, F1={best[3]:.3f})")
print("Compare to current hardcoded threshold (0.15) above to see if it's already near-optimal or should change.")

Using 197 labeled pairs (10 relevant, 187 not relevant)

Threshold   Precision   Recall      F1        
0.05        0.049       0.800       0.092     
0.1         0.125       0.800       0.216     
0.15        0.250       0.300       0.273      <- current (0.15)
0.2         0.667       0.200       0.308     
0.25        1.000       0.100       0.182     
0.3         1.000       0.100       0.182     
0.35        1.000       0.100       0.182     
0.4         0.000       0.000       0.000     

Best by F1: threshold=0.2 (precision=0.667, recall=0.200, F1=0.308)
Compare to current hardcoded threshold (0.15) above to see if it's already near-optimal or should change.
